
# Silver Layer — d_labitems

## Source
`cdr_mimiciv.bronze.d_labitems`

## Target
`cdr_mimiciv.silver.d_labitems`

The target table is published with CREATE OR REPLACE semantics. Downstream Gold
objects depend on it. No tables are dropped.

## Grain and key
One row per laboratory item, identified by `itemid` (official MIMIC-IV hosp
documentation, `d_labitems`). It has no dependency on any other Silver table.

## Source-file evidence (local MIMIC-IV CSV, 1,650 rows)
`itemid` is unique with no blanks. Four `label` values are empty and two have surrounding whitespace: `itemid` 51905
has the single-space label `' '` and `itemid` 52170 has `' Rbc'`. The trim / empty-to-NULL rule therefore changes exactly
those two non-empty labels (`' '` -> NULL, `' Rbc'` -> `'Rbc'`); the empty labels are already NULL when read.
The rule is whitespace standardisation only (see `docs/gold_governance_checks.md`, Source readiness).

## Purpose
Document and engineer the MIMIC-IV
laboratory item dictionary.

## Official source columns
- itemid
- label
- fluid
- category

## Engineering workflow
1. Understand the source
2. Profile data quality
3. Define transformation rules
4. Transform
5. Validate
6. Publish



In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

TARGET = f"{SILVER}.d_labitems"

# Load the Bronze source table
labitems_bronze = spark.table(
    f"{BRONZE}.d_labitems"
)

# Inspect its structure
print("=== BRONZE d_labitems ===")

print("Total rows:", labitems_bronze.count())
print("Total columns:", len(labitems_bronze.columns))
print("Column names:", labitems_bronze.columns)

# Inspect column datatypes
labitems_bronze.printSchema()

# Preview 10 records
display(labitems_bronze.limit(10))


In [0]:
%sql
Select distinct label
from cdr_mimiciv.bronze.d_labitems;

In [0]:
%sql
CREATE OR REPLACE TABLE cdr_mimiciv.silver.d_labitems AS
SELECT 
    itemid,
    category,

    -- 1. Create the Data Quality Flag 
    CASE 
        WHEN label IS NULL OR TRIM(label) = '' THEN FALSE
        WHEN label IN ('Delete', 'Voided Specimen', 'Problem Specimen', 'ZZDUMMY', 'SKIP', 'Not Done', 'reuse') THEN FALSE
        WHEN fluid IN ('I', 'Q') THEN FALSE -- Flag machine calibrations
        ELSE TRUE 
    END AS is_valid_clinical_test,

    -- 2. Standardize the Labels
    CASE 
        WHEN label IN ('Delete', 'Voided Specimen', 'Problem Specimen', 'ZZDUMMY', 'SKIP', 'Not Done', 'reuse') THEN 'System Artifact / Void'
        ELSE TRIM(REGEXP_REPLACE(SPLIT(label, ',')[0], '[()<>]+', ''))
    END AS label,

    -- 3. Clean and Standardize the Fluid Column
    CASE 
        WHEN fluid IN ('I', 'Q') THEN 'System Artifact / QC'
        WHEN fluid = 'Fluid' THEN 'Body Fluid (Generic)'
        WHEN fluid IS NULL OR TRIM(fluid) = '' THEN 'Unknown'
        ELSE TRIM(fluid)
    END AS fluid

FROM cdr_mimiciv.bronze.d_labitems;

In [0]:

# ==================================================
# STEP 3: BRONZE DATA QUALITY PROFILING
# Table: d_labitems
# ==================================================

# Check key uniqueness and missing values
profile = labitems_bronze.select(
    F.count("*").alias("total_rows"),

    F.countDistinct("itemid").alias("unique_itemids"),

    F.sum(F.when(F.col("itemid").isNull(), 1)
          .otherwise(0)).alias("null_itemids"),

    F.sum(F.when(F.col("label").isNull(), 1)
          .otherwise(0)).alias("null_labels"),

    F.sum(F.when(F.col("fluid").isNull(), 1)
          .otherwise(0)).alias("null_fluids"),

    F.sum(F.when(F.col("category").isNull(), 1)
          .otherwise(0)).alias("null_categories")
)

display(profile)


In [0]:

# ==================================================
# STEP 4: INVESTIGATE MISSING LABELS
# ==================================================

missing_labels = labitems_bronze.filter(
    F.col("label").isNull()
)

display(
    missing_labels.select(
        "itemid",
        "label",
        "fluid",
        "category"
    )
)


In [0]:

# STEP 5: TEXT QUALITY CHECK

text_columns = ["label", "fluid", "category"]

for c in text_columns:
    result = labitems_bronze.select(
        F.sum(
            F.when(F.trim(F.col(c)) == "", 1)
             .otherwise(0)
        ).alias("empty_strings"),

        F.sum(
            F.when(
                F.col(c) != F.trim(F.col(c)), 1
            ).otherwise(0)
        ).alias("whitespace_issues")
    )

    print(f"\nColumn: {c}")
    display(result)


In [0]:

# STEP 6: INSPECT TEXT QUALITY ISSUES

affected_labels = labitems_bronze.filter(
    F.col("label").isNotNull() &
    (
        (F.trim(F.col("label")) == "") |
        (F.col("label") != F.trim(F.col("label")))
    )
)

display(
    affected_labels.select(
        "itemid",
        F.concat(
            F.lit("["),
            F.col("label"),
            F.lit("]")
        ).alias("original_label"),
        F.length("label").alias("original_length"),
        F.trim("label").alias("trimmed_label"),
        "fluid",
        "category"
    )
)


In [0]:

# STEP 7: BUILD SILVER DATAFRAME

labitems_silver = (
    labitems_bronze
    .select(
        F.col("itemid"),

        F.when(
            F.trim(F.col("label")) == "",
            F.lit(None).cast("string")
        )
        .otherwise(F.trim(F.col("label")))
        .alias("label"),

        F.col("fluid"),
        F.col("category")
    )
)

# Preview transformed records
display(
    labitems_silver.filter(
        F.col("itemid").isin(51905, 52170)
    )
)


In [0]:

# STEP 8: VALIDATE SILVER TRANSFORMATION

validation = labitems_silver.select(
    F.count("*").alias("silver_rows"),

    F.countDistinct("itemid").alias("unique_itemids"),

    F.sum(
        F.when(F.col("itemid").isNull(), 1)
        .otherwise(0)
    ).alias("null_itemids"),

    F.sum(
        F.when(F.col("label").isNull(), 1)
        .otherwise(0)
    ).alias("null_labels"),

    F.sum(
        F.when(F.trim(F.col("label")) == "", 1)
        .otherwise(0)
    ).alias("empty_labels"),

    F.sum(
        F.when(
            F.col("label") != F.trim(F.col("label")), 1
        ).otherwise(0)
    ).alias("whitespace_issues")
)

display(validation)


In [0]:

# STEP 9: PRE-PUBLISH VALIDATION (on the Silver DataFrame)
# d_labitems has no dependency on other Silver tables.

labitems_final = labitems_silver.withColumn(
    "_silver_processed_at",
    F.current_timestamp()
)

bronze_rows = labitems_bronze.count()
pre = labitems_final.select(
    F.count("*").alias("rows"),
    F.countDistinct("itemid").alias("distinct_itemid"),
    F.sum(F.when(F.col("itemid").isNull(), 1).otherwise(0)).alias("null_itemid"),
    F.sum(F.when(F.trim(F.col("label")) == "", 1).otherwise(0)).alias("empty_labels"),
    F.sum(F.when(F.col("label") != F.trim(F.col("label")), 1).otherwise(0)).alias("untrimmed_labels"),
    F.sum(F.when(F.col("_silver_processed_at").isNull(), 1).otherwise(0)).alias("null_processed_at")
).first().asDict()
print("Bronze rows:", bronze_rows)
print("Pre-publish checks:", pre)

assert pre["rows"] == bronze_rows, "Silver transformation changed the row count"
assert pre["null_itemid"] == 0, "itemid contains NULLs"
assert pre["distinct_itemid"] == pre["rows"], "itemid is not unique"
assert pre["empty_labels"] == 0 and pre["untrimmed_labels"] == 0, "label text rule not applied"
assert pre["null_processed_at"] == 0


In [0]:

# STEP 10: PUBLISH d_labitems (CREATE OR REPLACE; Gold depends on this table)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    labitems_final.writeTo(TARGET)
    .using("delta")
    .createOrReplace()
)
print(f"Published: {TARGET}")

# Verify the published table
display(
    spark.table(TARGET).limit(10)
)


In [0]:

# STEP 11: FINAL PUBLISHED TABLE VALIDATION

published = spark.table(TARGET)

expected_schema = {
    "itemid": "int",
    "label": "string",
    "fluid": "string",
    "category": "string",
    "_silver_processed_at": "timestamp"
}
published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

post = published.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("itemid").alias("unique_itemids"),
    F.sum(F.when(F.col("itemid").isNull(), 1).otherwise(0)).alias("null_itemids"),
    F.sum(F.when(F.col("label").isNull(), 1).otherwise(0)).alias("null_labels"),
    F.sum(F.when(F.trim(F.col("label")) == "", 1).otherwise(0)).alias("empty_labels")
).first().asDict()

print("Published:", post)
print("Schema matches contract:", published_schema == expected_schema)

assert post["total_rows"] == bronze_rows, "Row count differs from Bronze"
assert post["null_itemids"] == 0
assert post["unique_itemids"] == post["total_rows"], "itemid is not unique"
assert post["empty_labels"] == 0
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"

dbutils.notebook.exit("d_labitems:OK")